[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-diabetes.ipynb)

# Full Project: Diabetes Risk Prediction (Healthcare)

*AIBits Academy · Machine Learning End To End · Full Project*

768 patients, a 7-model comparison in the classic Mathur "which algorithm wins?" style, and a genuine accuracy-vs-AUC disagreement that decides which model actually ships.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

def fetch(url, target, member=None):   # public source; a zip member is extracted and renamed to `target`
    if os.path.exists(target):
        return
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    blob = urllib.request.urlopen(req, timeout=120).read()
    if member:
        blob = zipfile.ZipFile(io.BytesIO(blob)).read(member)
    open(target, 'wb').write(blob)
    print('downloaded', target)

fetch('https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv', 'pima-indians-diabetes.data.csv')

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

**Load the Pima Indians diabetes data.** The file has no header row, so we name the eight features and the `Outcome` ourselves.

In [ ]:
cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv('pima-indians-diabetes.data.csv', header=None, names=cols)
print(df.shape, df['Outcome'].value_counts().to_dict())

> **Business Problem**
>
> A diagnostic lab chain wants a low-cost screening tool: given a patient's basic vitals from a routine check-up (already collected, no extra cost), flag which patients should be prioritised for the more expensive confirmatory HbA1c/glucose-tolerance test — without sending every patient for costly follow-up testing, and without missing genuine at-risk cases.

> **Dataset**
>
> **768 patients, 8 clinical features** — Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age — plus a binary `Outcome` (268 positive, 34.9% prevalence). [Dataset source →](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)

## Step 1 — A Data Quality Trap: Biologically Impossible Zeros

Before any modelling, a quick sanity check on the numeric columns reveals a classic real-world data issue:

In [ ]:
for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    print(col, (df[col]==0).sum())

A blood glucose, blood pressure, or BMI reading of exactly `0` is not a real measurement — it's how this particular data collection process encoded a missing value. **374 of 768 patients (49%)** are missing Insulin outright. Treating these as genuine zeros would badly distort every downstream model, so each is replaced with `NaN` then median-imputed:

In [ ]:
import numpy as np
zero_as_missing = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
for c in zero_as_missing:
    df[c] = df[c].replace(0, np.nan)
    df[c] = df[c].fillna(df[c].median())

## Step 2 — Seven-Model Comparison via 10-Fold Cross-Validation

Rather than committing to one algorithm upfront, seven classifiers are compared on identical, scaled, stratified train/test splits — the same "which family of algorithm even fits this problem" question every real project starts with:

After the zero-imputation, split (80/20, stratified) and standardise the features. The scaler is fitted on the training set only.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['Outcome'])
y = df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train_scaled, X_test_scaled = scaler.transform(X_train), scaler.transform(X_test)

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

models = [('LR',LogisticRegression(max_iter=1000)), ('LDA',LinearDiscriminantAnalysis()),
          ('KNN',KNeighborsClassifier()), ('CART',DecisionTreeClassifier(random_state=42)),
          ('NB',GaussianNB()), ('SVM',SVC(probability=True,random_state=42)),
          ('RFC',RandomForestClassifier(random_state=42,n_estimators=200))]

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
for name, model in models:
    scores = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='accuracy')
    print(f"{name}: {scores.mean():.4f} ({scores.std():.4f})")

Logistic Regression leads on cross-validated training accuracy (78.7%), with LDA close behind (77.9%) — both linear models outperforming the more flexible Decision Tree (70.5%), a common pattern on small-to-medium tabular datasets with a modest number of genuinely informative features.

## Step 3 — Held-Out Test Set: Accuracy and AUC Disagree

In [ ]:
for name, model in models:
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    proba = model.predict_proba(X_test_scaled)[:,1]
    print(f"{name}: acc={accuracy_score(y_test,preds):.4f}  auc={roc_auc_score(y_test,proba):.4f}")

On the held-out test set, **KNN wins on raw accuracy (75.3%)** but **Random Forest wins on AUC (0.8161)**, with Logistic Regression close behind on AUC (0.8130) despite a lower accuracy (70.8%). This is not a contradiction — it's the same ranking-vs-thresholded-decision distinction from the Model Evaluation page: accuracy is measured at one fixed 0.5 cutoff, while AUC measures ranking quality across every possible cutoff. For a screening tool whose entire purpose is to *rank* patients by risk and refer the highest-risk fraction for follow-up testing (an Expected-Value-style targeting decision, not a rigid yes/no at 0.5), AUC is the more relevant metric here — making Random Forest, not KNN, the better production choice despite its lower headline accuracy.

## Visualizing the Accuracy-vs-AUC Split

KNN's accuracy bar (gold ring) is tallest; Random Forest's AUC bar (gold ring) is tallest — two different winners on two different metrics, for the same 7 models.

> **⚠ A Health-Screening Tool Is Not a Diagnosis**
>
> Even the best AUC here (0.816) means this screening model is a genuinely useful triage signal, not a diagnostic replacement. In a real deployment, the confusion matrix and cost of a missed at-risk patient (a false negative sent home without follow-up testing) versus an unnecessary follow-up test (a false positive) would need the same Expected Value framework from the Model Evaluation page applied explicitly, with clinical input on the real cost of each error — not just an accuracy or AUC number chosen for its own sake.

## Step 4 — Which Features Actually Drive Risk?

The random forest fitted in the previous cell's loop is the one whose importances we read.

In [ ]:
rfc = dict(models)['RFC']

In [ ]:
importances = pd.Series(rfc.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

Glucose dominates at nearly 27% of total importance — unsurprising clinically, since fasting glucose is itself the basis of a diabetes diagnosis, but reassuring that the model has learned a medically sensible signal rather than an arbitrary correlation. BMI and family history (DiabetesPedigreeFunction) follow as the next-strongest signals, consistent with established risk-factor literature.

## Key Business Takeaways

- 49% of patients were missing an Insulin reading (encoded as an impossible zero) — a data-quality check that must run before any model is trusted, not after.
- Comparing 7 algorithms head-to-head revealed a genuine accuracy-vs-AUC split: KNN wins accuracy (75.3%), Random Forest wins AUC (0.816) — and for a risk-ranking screening tool, AUC is the metric that should decide deployment.
- Glucose, BMI, and family history collectively drive over half of the Random Forest's predictive signal, matching established clinical risk factors — a useful sanity check that the model learned something medically plausible.

## Step 5 — Deployment: From Trained Model to a Usable App

A model that only lives inside a notebook helps no one. The final step of any real project is **deployment** — packaging the trained model so a non-technical user (here, a lab technician) can feed in a new patient's vitals and get an instant risk flag, without ever touching Python. This is exactly the boundary where a data scientist's work meets the emerging **MLOps / full-stack data scientist** role.

> **📦 The Two Halves of Deployment**
>
> Deployment splits into two independent steps: **(1) serialize** the trained model to a file once, and **(2) load** that file inside a lightweight app that serves predictions. The heavy training never runs again — the app just loads the frozen model and calls `.predict()`.

### Step 5a — Serialize the Model (Pickle / Joblib)

Python's `pickle` writes any Python object — including a fully-trained scikit-learn model — to a binary file. For scikit-learn models specifically, `joblib` is the officially recommended equivalent (it stores the large internal NumPy arrays more efficiently). We freeze the Random Forest that won on AUC in Step 3:

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier

# rf is the model already trained on the cleaned Pima data (Steps 1-3)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)

# Freeze it to disk once — this is the artifact you deploy
joblib.dump(rf, 'diabetes_rf.joblib')
print("Model saved.")

# Later, in a completely separate process, load it back:
loaded = joblib.load('diabetes_rf.joblib')
print("Reloaded predictions identical to original:",
      (loaded.predict(X_test) == rf.predict(X_test)).all())

The reloaded model is byte-for-byte the same predictor — no retraining, no loss of accuracy. The saved file here is about **3.3 MB** (Random Forests store every tree; a Logistic Regression would be a few KB). That single file is now the deployable artifact.

> **⚠ Pickle Security & Version Pinning**
>
> Two real-world gotchas: (1) **Never unpickle a file from an untrusted source** — a malicious pickle can execute arbitrary code on load. (2) A model pickled under one scikit-learn version may warn or break when loaded under a different one, so **pin the library version** (e.g. record `scikit-learn==1.8.0` alongside the artifact) and retrain if you upgrade.

### Step 5b — Serve Predictions with a Streamlit App

**Streamlit** turns a plain Python script into an interactive web app with no HTML/JS/CSS required — ideal for a data scientist who wants to ship a usable tool fast. The app below loads the frozen model once and gives the lab technician a form: type in a patient's vitals, click a button, get a risk verdict and probability.

> **Run this one on your own computer (terminal or local Jupyter), not in Colab** (it is a shell command, or needs a desktop window, a running server, or keyboard input).

```python
# app.py  —  run locally with:  streamlit run app.py
import streamlit as st
import pandas as pd
import joblib

model = joblib.load('diabetes_rf.joblib')   # loaded once, cached by Streamlit

st.title('Diabetes Risk Screening Tool')
st.write('Enter a patient\'s routine check-up vitals to get a risk flag.')

# One input widget per feature — order MUST match training columns
preg    = st.number_input('Pregnancies', 0, 20, 1)
glucose = st.number_input('Glucose', 0, 300, 120)
bp      = st.number_input('BloodPressure', 0, 200, 70)
skin    = st.number_input('SkinThickness', 0, 100, 20)
insulin = st.number_input('Insulin', 0, 900, 80)
bmi     = st.number_input('BMI', 0.0, 70.0, 30.0)
dpf     = st.number_input('DiabetesPedigreeFunction', 0.0, 3.0, 0.4)
age     = st.number_input('Age', 18, 100, 33)

if st.button('Assess Risk'):
    patient = pd.DataFrame([[preg, glucose, bp, skin, insulin, bmi, dpf, age]],
                           columns=['Pregnancies','Glucose','BloodPressure',
                                    'SkinThickness','Insulin','BMI',
                                    'DiabetesPedigreeFunction','Age'])
    proba = model.predict_proba(patient)[0][1]
    if model.predict(patient)[0] == 1:
        st.error(f'⚠ High risk — prioritise for HbA1c test. (p = {proba:.1%})')
    else:
        st.success(f'✓ Low risk. (p = {proba:.1%})')
```

Running `streamlit run app.py` launches a local web server (default `localhost:8501`) with a live form. Two example patients, scored by the actual loaded model:

| Patient profile | Model verdict | P(diabetic) |
|---|---|---|
| Glucose 168, BMI 38.5, Age 51, pedigree 0.62 (high-risk profile) | ⚠ High risk | 0.815 |
| Glucose 95, BMI 24.1, Age 25, pedigree 0.18 (low-risk profile) | ✓ Low risk | 0.005 |

The same frozen `diabetes_rf.joblib` could equally be wrapped in a **Flask/FastAPI REST endpoint** (for programmatic access by a hospital's existing software) instead of a Streamlit form — the serialize-then-load pattern is identical; only the serving layer changes.

> **🚀 Beyond Deployment — The MLOps Frontier**
>
> Shipping the app is not the end. A production model needs **monitoring** (is live accuracy drifting as patient demographics shift?), **periodic retraining** on fresh data, and **versioned rollbacks** when a new model underperforms — the *Model Drift Analysis* and *recalibration* loop. Those responsibilities define the **full-stack data scientist / ML engineer** role and are a course in their own right, deferred here to a dedicated future **MLOps** course. This page takes you as far as a working, deployable prediction tool.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Stratified splits keep the prevalence

Store the positive rate of `y_train` in `pos_train` and of `y_test` in `pos_test`. Because we stratified, they should be within 0.01 of each other.

In [ ]:
pos_train = pos_test = None   # TODO


In [ ]:
try:
    check("about 34.9%", abs(pos_train - 0.349) < 0.01)
    check("test matches train", abs(pos_train - pos_test) < 0.01)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
pos_train = float(y_train.mean())
pos_test = float(y_test.mean())

```

</details>

### Exercise 2 · Medium · Impossible zeros

Re-read the file into `raw`. Count how many rows have a biologically impossible **zero** in at least one of `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`, and store it in `n_bad_rows`.

In [ ]:
raw = pd.read_csv("pima-indians-diabetes.data.csv", header=None, names=cols)
n_bad_rows = None   # TODO


In [ ]:
try:
    ref = int((raw[["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]] == 0).any(axis=1).sum())
    check("count", n_bad_rows == ref)
    check("about half the rows", 300 < n_bad_rows < 400)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
raw = pd.read_csv("pima-indians-diabetes.data.csv", header=None, names=cols)
n_bad_rows = int((raw[["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]] == 0).any(axis=1).sum())

```

</details>

### Exercise 3 · Stretch · Screening needs recall

For the logistic-regression model in `models`, compare recall at the default 0.5 threshold with recall at 0.35. Store them in `recall_05` and `recall_035`; lowering the threshold should catch more diabetics.

In [ ]:
recall_05 = recall_035 = None   # TODO (models were fitted in the last loop)


In [ ]:
try:
    check("lower threshold has higher recall", recall_035 > recall_05)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
lr_model = dict(models)["LR"]
p = lr_model.predict_proba(X_test_scaled)[:, 1]
recall_05 = recall_score(y_test, p >= 0.5)
recall_035 = recall_score(y_test, p >= 0.35)

```

In screening, a missed diabetic costs far more than an extra test, so the threshold should sit below 0.5.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Diabetes Risk Prediction (Healthcare)**.*